## Agent Evaluation

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [10]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [7]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [ ]:
# Message history
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course late — can I still sign up and take part?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late registration signup join course after start take part"}', call_id='call_QmfSVFguzu0A2tfj7t2lpptq', name='search', type='function_call', id='fc_07c3ef54e8638812006a58f4065c8881a2bb56f8cf00e55dec', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_QmfSVFguzu0A2tfj7t2lpptq',
  'output': '[\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accep

In [9]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [ ]:
# Our trajectory - only tool calls
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late registration signup join course after start take part"}'}]

In [11]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [ ]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

# AQA' record + trajectory (our set for agent evaluation)
agent_result

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_agent': 'Yes — you can still join the course even if you found it late.\n\nFrom the FAQ: **you can start learning and participating whenever you want**. If you want a **certificate**, the key thing is to **submit your project while submissions are still open**.\n\nIf you’d like, I can also point you to the recommended “start here” resources for the course.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late registration signup join course after start take part"}'}],
 'cost': Decimal('0.00119925'),
 'document': '74eb249bbf'}

In [14]:
# Wrap above into function
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Execute for smaller sample
with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [16]:
df_agent = pd.DataFrame(agent_answers)
df_agent["cost"].sum()

Decimal('0.06425925')

In [17]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [ ]:
# Agent Evaluation set: AQA' + trajectory (tool calls) - to be validated by LLM judge
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course late — can I still si...,Yes — you can still join the course even if yo...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.00117375,74eb249bbf
1,Am I allowed to join the course after it alrea...,Yes — you can join even after the course has s...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""jo...",0.00128175,74eb249bbf
2,"If I start the course now, is there still a ch...","Yes — there can still be a chance, but only if...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00113325,74eb249bbf
3,What do I need to do to be eligible for a cert...,"If you joined late, you can still be eligible ...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00114825,74eb249bbf
4,"Is late enrollment okay, or is the project dea...",Late submissions are **not allowed once the su...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.00113175,74eb249bbf


## Evaluate

In [3]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

# LLM judge output format for Agent Evaluation 
# (we add 2 output fields when compared to RAG)
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [5]:
# LLM judge prompt template

agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [6]:
# Evaluation flow function (creates prompt and calls LLM)
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [8]:
agent_answers[0]

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_agent': 'Yes — you can still join the course even if you discovered it late.\n\nThe only important caveat is that if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"late enrollment sign up participate course late found course can I still sign up and take part"}\'}]',
 'cost': 0.00117375,
 'document': '74eb249bbf'}

In [11]:
import ast

# Test on 1 result
rec = agent_answers[0].copy()
tool_calls = rec["tool_calls"]

if isinstance(tool_calls, str):
    try:
        rec["tool_calls"] = json.loads(tool_calls)
    except json.JSONDecodeError:
        rec["tool_calls"] = ast.literal_eval(tool_calls)

agent_eval, usage = evaluate_agent_answer(rec)

agent_eval

AgentEvaluation(answer_reasoning='The agent answer matches the ground truth: it says the student can still join the course, and that to receive a certificate they must submit the project while submissions are still open. This preserves the key information and wording closely enough.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included the core idea of late enrollment/signing up after discovering the course. A single search call was sufficient and there were no unnecessary or duplicate calls.', trajectory_score='good')

In [13]:
agent_answers[0]

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_agent': 'Yes — you can still join the course even if you discovered it late.\n\nThe only important caveat is that if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"late enrollment sign up participate course late found course can I still sign up and take part"}\'}]',
 'cost': 0.00117375,
 'document': '74eb249bbf'}

In [14]:
rec

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_agent': 'Yes — you can still join the course even if you discovered it late.\n\nThe only important caveat is that if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late enrollment sign up participate course late found course can I still sign up and take part"}'}],
 'cost': 0.00117375,
 'document': '74eb249bbf'}

In [ ]:
def judge_agent_record(rec):
    tool_calls = rec["tool_calls"]

    # Fix non-parsable JSON
    if isinstance(tool_calls, str):
        try:
            rec["tool_calls"] = json.loads(tool_calls)
        except json.JSONDecodeError:
            rec["tool_calls"] = ast.literal_eval(tool_calls)

    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [20]:
# Split the results
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [21]:
df_agent_eval = pd.DataFrame(agent_evaluations)
calc_total_price(usages)

0.053889000000000006

In [24]:
# Anwsers score
df_agent_eval["answer_score"].value_counts()

answer_score
good    46
bad      4
Name: count, dtype: int64

In [25]:
# Trajectories score
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [26]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)